# Insurance Risk Analytics

# Task 4: Statistical Modeling & Risk-Based Pricing

## Business Objective

The objective of this task is to develop predictive models that estimate insurance claim severity. The best-performing model will support risk-based pricing by helping insurers better estimate expected claim costs and improve premium pricing decisions.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append("../")
import sklearn
print(sklearn.__version__)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from src.config import (
    DATA_PATH,
    MODELS_PATH
)

from src.data_loader import (
    load_data,
    preprocess_data
)

from src.modeling import (
    prepare_data,
    select_features,
    split_data,
    build_preprocessor,
    train_model,
    evaluate_model,
    compare_models,
    save_model,
)

1.0.2


In [2]:
df = load_data(DATA_PATH)

df = preprocess_data(df)

df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,TransactionYear,VehicleAge
0,145249,12827,2015-03-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,2015,11
1,145249,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,2015,11
2,145249,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,2015,11
3,145255,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,2015,11
4,145255,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,2015,11


In [3]:
X, y = prepare_data(df)

X = select_features(X)
X.head()

,VehicleAge,VehicleType,make,Model,bodytype,kilowatts,cubiccapacity,Province,Gender,MaritalStatus,CoverType,CoverCategory,TermFrequency,SumInsured,CalculatedPremiumPerTerm,CustomValueEstimate,CapitalOutstanding
203,3,Passenger Vehicle,AUDI,A4 1.8T ATTRACTION (B8),S/D,118.0,1781.0,Gauteng,Not specified,Not specified,Own Damage,Own damage,Monthly,208800.00,891.8912,NaN,208800
284,9,Passenger Vehicle,FORD,TERRITORY 4.0i GHIA AWD A/T,SUV,182.0,3984.0,Gauteng,Not specified,Not specified,Windscreen,Windscreen,Monthly,0.01,25.0000,NaN,127300
1560,1,Passenger Vehicle,BMW,316i A/T (F30),S/D,100.0,1598.0,Gauteng,Not specified,Not specified,Own Damage,Own damage,Monthly,408000.00,1383.8337,408000.0,0
1779,5,Medium Commercial,VOLKSWAGEN,CRAFTER 50 HR 80 F/C P/V,P/V,80.0,2459.0,KwaZulu-Natal,Not specified,Not specified,Own Damage,Own Damage,Monthly,206900.00,735.3199,NaN,0
1943,1,Passenger Vehicle,BMW,316i A/T (F30),S/D,100.0,1598.0,Gauteng,Not specified,Not specified,Income Protector,Income Protector,Monthly,7000.00,85.0000,408000.0,0


In [4]:
X.columns

Index(['VehicleAge', 'VehicleType', 'make', 'Model', 'bodytype', 'kilowatts',
       'cubiccapacity', 'Province', 'Gender', 'MaritalStatus', 'CoverType',
       'CoverCategory', 'TermFrequency', 'SumInsured',
       'CalculatedPremiumPerTerm', 'CustomValueEstimate',
       'CapitalOutstanding'],
      dtype='object')

In [5]:
X["VehicleAge"].describe()

count    2788.000000
mean        4.409254
std         2.967957
min         0.000000
25%         2.000000
50%         4.000000
75%         7.000000
max        18.000000
Name: VehicleAge, dtype: float64

In [6]:
X_train, X_test, y_train, y_test = split_data(X, y)

In [7]:
preprocessor = build_preprocessor(X)

### Linear Regression

In [8]:
linear_model = train_model(
    LinearRegression(),
    preprocessor,
    X_train,
    y_train
)

linear_rmse, linear_r2 = evaluate_model(
    linear_model,
    X_test,
    y_test
)

### Random Forest

In [9]:
rf_model = train_model(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    preprocessor,
    X_train,
    y_train
)

rf_rmse, rf_r2 = evaluate_model(
    rf_model,
    X_test,
    y_test
)

### XGBoost

In [10]:
xgb_model = train_model(
    XGBRegressor(
        random_state=42,
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6
    ),
    preprocessor,
    X_train,
    y_train
)

xgb_rmse, xgb_r2 = evaluate_model(
    xgb_model,
    X_test,
    y_test
)

### Compare Models

In [11]:
results = [

    {
        "Model": "Linear Regression",
        "RMSE": linear_rmse,
        "R²": linear_r2
    },

    {
        "Model": "Random Forest",
        "RMSE": rf_rmse,
        "R²": rf_r2
    },

    {
        "Model": "XGBoost",
        "RMSE": xgb_rmse,
        "R²": xgb_r2
    }

]

comparison = compare_models(results)

comparison

,Model,RMSE,R²
0,Linear Regression,36960.767028,0.150567
1,Random Forest,37088.525913,0.144685
2,XGBoost,37227.387355,0.138268


In [16]:
best_model_name = comparison.iloc[0]["Model"]

if best_model_name == "Linear Regression":
    best_model = linear_model

elif best_model_name == "Random Forest":
    best_model = rf_model

else:
    best_model = xgb_model

MODELS_PATH.mkdir(exist_ok=True)

save_model(
    best_model,
    MODELS_PATH / "best_model.pkl"
)

print(f"Best Model: {best_model_name}")

Best Model: Linear Regression


In [17]:
best_model = linear_model

### Global Importance Plot

In [14]:
X,y = prepare_data(df)

X = select_features(X)


X_train,X_test,y_train,y_test = split_data(
    X,y
)


preprocessor = build_preprocessor(
    X_train
)


linear_model = train_model(
    LinearRegression(),
    preprocessor,
    X_train,
    y_train
)


rmse,r2 = evaluate_model(
    linear_model,
    X_test,
    y_test
)


print(rmse,r2)

36960.767028192015 0.1505673823428293


In [15]:
import joblib

model = joblib.load(
    "../models/best_model.pkl"
)

print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['VehicleAge', 'kilowatts',
                                                   'cubiccapacity',
                                                   'SumInsured',
                                                   'CalculatedPremiumPerTerm',
                                                   'CustomValueEstimate']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                        